# Linear Algebra

Linear algebra is the branch of mathematics that deals with *vector spaces*. 

## Vectors
*Vectors* are objects that can be added together to form new vectors and that can be multiplied by *scalars* (i.e., numbers), also to form new vectors. Vectors are points in some finite-dimensional space. Useful to represent numeric data. For example, if you have the heights, weights, and ages of a large number of people, you can treat your data as three-dimensional vectors ``height``, ``weight``, ``age]``. If you’re teaching a class with four exams, you can treat
student grades as four-dimensional vectors [``exam1``, ``exam2``, ``exam3``,``exam4``].

The simplest from-scratch approach is to represent vectors as lists of numbers. A list of three numbers corresponds to a vector in three-dimensional space, and vice versa. 

We'll accomplish this with a type alias that says a ```Vector``` is just a ```list``` of ```floats```:

In [256]:
from typing import List

Vector = List[float]

height_weight_age = [70,    # inches 
                    170,    # pounds
                    40]     # year

# exam1, exam2, ...
grades = [95, 80, 75, 62]

We'll also perform *arithmetic* on vectors. Sicne ```list```s arent vectors, we need to build arithmetic tools. 

Adding two vectors *componentwise*. This means that if two vectors ``v`` and ``w`` are the same length, their sum is just the vector whose first element is ``v[0]`` + ``w[0]``, whose second element is ``v[1]`` + ``w[1]``, and so on. 

We can easily implement this by ``zip``-ing the vectors together and using a list comprehension to add the corresponding elements:

In [257]:
from typing import List

Vector = List[float]

def add(v: Vector, w: Vector) -> Vector:
    """Adds corresponding elements"""
    assert len(v) == len(w), "vectors must be the same length"
    return [v_i + w_i for v_i, w_i in zip(v, w)]

assert add([1, 2, 3], [4, 5, 6]) == [5, 7, 9]

Similarly, to subtract two vectors we just subtract the corresponding elements:

In [258]:
def subtract(v: Vector, w: Vector) -> Vector:
    """Subtracts corresponding elements"""
    assert len(v) == len(w), "vectors must be the same length"
    return [v_i - w_i for v_i, w_i in zip(v, w)]

assert subtract([5, 7, 9], [4, 5, 6]) == [1, 2, 3]

We'll sometimes want to componentwise sum a list of vectors - that is create a new vector whose first elements is the sum of all the first elements, whose second element is the sum of all the second elements, and so on:

In [259]:
def vector_sum(vectors: List[Vector]) -> Vector:
    """Sums all corresponding elements"""
    # Check that vectors is not empty
    assert vectors, "no vector provided"
    
    # Check the vectors are all the same size
    num_elements = len(vectors[0])
    assert all(len(v) == num_elements for v in vectors), "different sizes"
    
    # the i-th element of the result is the sum of every vector[i]
    return [sum(vector[i] for vector in vectors) for i in range(num_elements)]
assert vector_sum([[1, 2], [3, 4], [5, 6], [7, 8]]) == [16, 20]

We'll also need to be able to multiply a vector by a scalar, which we do simply by multiplying each element of the vector by that number:

In [260]:
def scalar_multiply(c: float, v: Vector) -> Vector:
    """Mulitplies every element by c"""
    return [c * v_i for v_i in v]

assert scalar_multiply(2, [1, 2, 3]) == [2, 4, 6]



This allows us to compute the componenentwise means of a list of (same sized) vectors:

In [261]:
def vector_mean(vectors: List[Vector]) -> Vector:
    """Computes the element-wise average"""
    n = len(vectors)
    return scalar_multiply(1/n, vector_sum(vectors))

assert vector_mean([[1, 2], [3, 4], [5, 6]]) == [3, 4]    

A less obvious tool is the *dot* product. The dot product of two vectors is the sum of their componenetwise products:

In [262]:
def dot(v: Vector, w: Vector) -> float:
    """Computes v_1 * w_1 + ... + v_n * w_n"""
    assert len(v) == len(w), "vectors must be the same length"
    
    return sum(v_i * w_i for v_i, w_i in zip(v, w))

assert dot([1, 2, 3], [4, 5, 6]) == 32 # 1 * 4 + 2 * 5 + 3 * 6 = 32

If ```w``` has magnitude 1, the dot product measures how far the vector ```v``` expands in the ```w``` direction. For example, if ```w = [1, 0]```, then ```dot(v, w)``` is just the first component of ```v```. Another way of saying this is that it's the length of the vector you'd get if you *projected* ```v``` onto ```w```

In [263]:
def sum_of_squares(v: Vector) -> float:
    """Returns v_1 * v_1 + ... + v_n * v_n"""
    return dot(v, v)

assert sum_of_squares([1, 2, 3]) == 14

We can use this to compute the magnitude (length)

In [264]:
import math

def magnitude(v: Vector) -> float:
    """Returns the magnitude (or length) of vector v"""
    return math.sqrt(sum_of_squares(v))

assert magnitude([3, 4]) == 5

Now we have the pieces to compute the distance between two vectors, defined as:

$$\sqrt{(v_1 - w_1)^2 + . . . + (v_n - w_n)^2}$$

In code:

In [265]:
def squared_distances(v: Vector, w: Vector) -> float:
    """Computes (v_1 - w_1) ** 2 + ... + (v_n - w_n) ** 2"""
    return sum_of_squares(subtract(v, w))

def distance(v: Vector, w: Vector) -> float:
    """Computes the distance between v and w"""
    return math.sqrt(squared_distances(v, w))

## Matrices

A *matrix* is a two-dimensional collection of numbers. We will represent matrices as lists of lists, with each inner list having the same size and representing a *row* of the matrix. If ```A``` is a matrix, then ```A[i][j]``` is the element in the *i*th row and the *j*th column. 

In [266]:
# Another type alias
Matrix = List[List[float]]

A = [[1, 2, 3],  # A has 2 rows and 3 columns
     [4, 5, 6]] 

B = [[1, 2],
    [3, 4],
    [5, 6]]

Given the lists-of-lists representation, the matrix ```A``` has ```len(A)``` rows and ```len(A[0])``` columns, which we consider its ```shape```:

In [267]:
from typing import Tuple
def shape(A: Matrix) -> Tuple[int, int]:
    """Returns the (# of rows of A, # of columns of A)"""
    num_rows = len(A)
    num_cols = len(A[0]) if A else 0 # number of elements in the first row
    
    return num_rows, num_cols

assert shape([[1, 2, 3], [4, 5, 6]]) == (2, 3)

If a matrix has ```n``` rows and ```k``` columns, we will refer to it as an *n* x *k* matrix. We can think of each row of an  *n* x *k* matrix as a vector of length *k*, and each columns as a vector of length *n*:

In [268]:
def get_row(A: Matrix, i: int) -> Vector:
    """Returns the i-th row of A (as a vector)"""
    return A[i]

def get_row(A: Matrix, j: int) -> Vector:
    "Returns the j-th col of A (as a vector)"
    column = [A_i[j] for A_i in A]
    return column

We'll also want to be able to create a matrix given its shape and a function for generating its elements. We can do this as a nested list of comprehension:

In [269]:
from typing import Callable

def make_matrix(num_rows: int,
                num_cols: int,
                entry_fn: Callable[[int, int], float]) -> Matrix:
    """Returns a num_rows x num_cols matrix
    whose (i, j)-th entry is entry_fn(i, j)."""

    return [
        [entry_fn(i, j) for j in range(num_cols)]
        for i in range(num_rows)
    ]

Given this function, you could make a $5 \ \text{x} \ 5$ *identity matrix* (wiht 1s on the diagonal and 0s elsewhere) like so:

In [270]:
def identity_matrix(n: int) -> Matrix:
    """Returns the n x n identity matrix"""
    return make_matrix(n, n, lambda i, j: 1 if i == j else 0)
assert identity_matrix(5) == [[1, 0, 0, 0, 0],
                            [0, 1, 0, 0, 0],
                            [0, 0, 1, 0, 0],
                            [0, 0, 0, 1, 0],
                            [0, 0, 0, 0, 1]]

Matrices are good for representing binary relationships. For example we represented the edges of a network as a collection of pairs ```(i, j)``` . An alternative representation would be to create a matrix ```A``` such that ```[A][i][j]``` is 1 if nodes ```i``` and  ```j``` are connected and 0 otherwise. 

Before we had:

In [271]:
friendships = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4),
                (4, 5), (5, 6), (5, 7), (6, 8), (7, 8), (8, 9)]


This could also be represented as:



In [272]:

# user 0 1 2 3 4 5 6 7 8 9
friend_matrix = [[0, 1, 1, 0, 0, 0, 0, 0, 0, 0], # user 0
[1, 0, 1, 1, 0, 0, 0, 0, 0, 0], # user 1
[1, 1, 0, 1, 0, 0, 0, 0, 0, 0], # user 2
[0, 1, 1, 0, 1, 0, 0, 0, 0, 0], # user 3
[0, 0, 0, 1, 0, 1, 0, 0, 0, 0], # user 4
[0, 0, 0, 0, 1, 0, 1, 1, 0, 0], # user 5
[0, 0, 0, 0, 0, 1, 0, 0, 1, 0], # user 6
[0, 0, 0, 0, 0, 1, 0, 0, 1, 0], # user 7
[0, 0, 0, 0, 0, 0, 1, 1, 0, 1], # user 8
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0]] # user 9

If there are very few connections, this is a much more inefficient
representation, since you end up having to store a lot of zeros. However,
with the matrix representation it is much quicker to check whether two
nodes are connected—you just have to do a matrix lookup instead of
(potentially) inspecting every edge:

In [273]:
assert friend_matrix[0][2] == 1, "0 and 2 are friends"
assert friend_matrix[0][8] == 0, "0 and 8 are not friends"

Similarly, to find a node’s connections, you only need to inspect the column (or the row) corresponding to that node:

In [274]:
# only need to look at one row
friends_of_five = [i for i, is_friend in enumerate(friend_matrix[5]) if is_friend]

With a small graph you could just add a list of connections to each node object to speed up this process; but for a large, evolving graph that would probably be too expensive and difficult to maintain.

In [275]:
import os
print(os.getcwd())

/Users/bmart231/DS/Notebooks/04-LinAlg


In [276]:
import os

for root, dirs, files in os.walk("../"):
    print(root)

../
../03-Visualizing_Data
../02-CrashCourse
../06-Probability
../01-Introduction
../05-Stats
../04-LinAlg
